# COMS W4705 Spring 25
## Homework 4 - Semantic Role Labelling with BERT

The goal of this assignment is to train and evaluate a PropBank-style semantic role labeling (SRL) system. Following (Collobert et al. 2011) and others, we will treat this problem as a sequence-labeling task. For each input token, the system will predict a B-I-O tag, as illustrated in the following example:

|The|judge|scheduled|to|preside|over|his|trial|was|removed|from|the|case|today|.|             
|---|-----|---------|--|-------|----|---|-----|---|-------|----|---|----|-----|-|             
|B-ARG1|I-ARG1|B-V|B-ARG2|I-ARG2|I-ARG2|I-ARG2|I-ARG2|O|O|O|O|O|O|O|
|||schedule.01|||||||||||||

Note that the same sentence may have multiple annotations for different predicates

|The|judge|scheduled|to|preside|over|his|trial|was|removed|from|the|case|today|.|             
|---|-----|---------|--|-------|----|---|-----|---|-------|----|---|----|-----|-|             
|B-ARG1|I-ARG1|I-ARG1|I-ARG1|I-ARG1|I-ARG1|I-ARG1|I-ARG1|O|B-V|B-ARG2|I-ARG2|I-ARG2|B-ARGM-TMP|O|
||||||||||remove.01||||||

and not all predicates need to be verbs

|The|judge|scheduled|to|preside|over|his|trial|was|removed|from|the|case|today|.|             
|---|-----|---------|--|-------|----|---|-----|---|-------|----|---|----|-----|-|    
|O|O|O|O|O|O|B-ARG1|B-V|O|O|O|O|O|O|O|
||||||||try.02||||||||

The SRL system will be implemented in [PyTorch](https://pytorch.org/). We will use BERT (in the implementation provided by the [Huggingface transformers](https://huggingface.co/docs/transformers/index) library) to compute contextualized token representations and a custom classification head to predict semantic roles. We will fine-tune the pretrained BERT model on the SRL task.


### Overview of the Approach

The model we will train is pretty straightforward. Essentially, we will just encode the sentence with BERT, then take the contextualized embedding for each token and feed it into a classifier to predict the corresponding tag.

Because we are only working on argument identification and labeling (not predicate identification), it is essentially that we tell the model where the predicate is. This can be accomplished in various ways. The approach we will choose here repurposes Bert's *segment embeddings*.

Recall that BERT is trained on two input sentences, seperated by [SEP], and on a next-sentence-prediction objective (in addition to the masked LM objective). To help BERT comprehend which sentence a given token belongs to, the original BERT uses a segment embedding, using A for the first sentene, and B for the second sentence 2.
Because we are labeling only a single sentence at a time, we can use the segment embeddings to indicate the predicate position instead: The predicate is labeled as segment B (1) and all other tokens will be labeled as segment A (0).

<img src="https://github.com/daniel-bauer/4705-f23-hw5/blob/main/bert_srl_model.png?raw=true" width=400px>

## Setup: GCP, Jupyter, PyTorch, GPU

To make sure that PyTorch is available and can use the GPU,run the following cell which should return True. If it doesn't, make sure the GPU drivers and CUDA are installed correctly.

GPU support is required for this assignment -- you will not be able to fine-tune BERT on a CPU.

In [ ]:
import torch
torch.cuda.is_available()

True

## Dataset: Ontonotes 5.0 English SRL annotations

We will work with the English part of the [Ontonotes 5.0](https://catalog.ldc.upenn.edu/LDC2013T19) data. This is an extension of PropBank, using the same type of annotation. Ontonotes contains annotations other than predicate/argument structures, but we will use the PropBank style SRL annotations only. *Important*: This data set is provided to you for use in COMS 4705 only! Columbia is a subscriber to LDC and is allowed to use the data for educational purposes. However, you may not use the dataset in projects unrelated to Columbia teaching or research.

If you haven't done so already, you can download the data here:


In [ ]:
! wget https://storage.googleapis.com/4705-bert-srl-data/ontonotes_srl.zip

--2025-05-01 20:47:10--  https://storage.googleapis.com/4705-bert-srl-data/ontonotes_srl.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 74.125.24.207, 142.251.10.207, 142.251.12.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|74.125.24.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12369688 (12M) [application/zip]
Saving to: ‘ontonotes_srl.zip’

ontonotes_srl.zip   100%[===================>]  11.80M  5.73MB/s    in 2.1s    

2025-05-01 20:47:13 (5.73 MB/s) - ‘ontonotes_srl.zip’ saved [12369688/12369688]



In [ ]:
! unzip ontonotes_srl.zip

Archive:  ontonotes_srl.zip
  inflating: propbank_dev.tsv        
  inflating: propbank_test.tsv       
  inflating: propbank_train.tsv      
  inflating: role_list.txt           


The data has been pre-processed in the following format. There are three files:

`propbank_dev.tsv`	`propbank_test.tsv`	`propbank_train.tsv`

Each of these files is in a tab-separated value format. A single predicate/argument structure annotation consists of four rows. For example

```
ontonotes/bc/cnn/00/cnn_0000.152.1
The     judge   scheduled       to      preside over    his     trial   was     removed from    the     case    today   /.
                schedule.01
B-ARG1  I-ARG1  B-V     B-ARG2  I-ARG2  I-ARG2  I-ARG2  I-ARG2  O       O       O       O       O       O       O
```

* The first row is a unique identifier (1st annotation of the 152nd sentence in the file ontonotes/bc/cnn/00/cnn_0000).
* The second row contains the tokens of the sentence (tab-separated).
* The third row contains the probank frame name for the predicate (empty field for all other tokens).
* The fourth row contains the B-I-O tag for each token.

The file `rolelist.txt` contains a list of propbank BIO labels in the dataset (i.e. possible output tokens). This list has been filtered to contain only roles that appeared more than 1000 times in the training data.
We will load this list and create mappings from numeric ids to BIO tags and back.

In [ ]:
role_to_id = {}
with open("role_list.txt",'r') as f:
    role_list = [x.strip() for x in f.readlines()]
    role_to_id = dict((role, index) for (index, role) in enumerate(role_list))
    role_to_id['[PAD]'] = -100

    id_to_role = dict((index, role) for (role, index) in role_to_id.items())


Note that we are also mapping the '[PAD]' token to the value -100. This allows the loss function to ignore these tokens during training.

## Part 1 - Data Preparation

Before you can build the SRL model, you first need to preprocess the data.


### 1.1 - Tokenization

One challenge is that the pre-trained BERT model uses subword ("WordPiece") tokenization, but the Ontonotes data does not. Fortunately Huggingface transformers provides a tokenizer.

In [ ]:
from transformers import BertTokenizerFast
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased', do_lower_case=True)
tokenizer.tokenize("This is an unbelievably boring test sentence.")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

['this',
 'is',
 'an',
 'un',
 '##bel',
 '##ie',
 '##va',
 '##bly',
 'boring',
 'test',
 'sentence',
 '.']

**TODO**:
We need to be able to maintain the correct labels (B-I-O tags) for each of the subwords.
Complete the following function that takes a list of tokens and a list of B-I-O labels of the same length as parameters, and returns a new token / label pair, as illustrated in the following example.


```
>>> tokenize_with_labels("the fancyful penguin devoured yummy fish .".split(), "B-ARG0 I-ARG0 I-ARG0 B-V B-ARG1 I-ARG1 O".split(), tokenizer)
(['the',
  'fancy',
  '##ful',
  'penguin',
  'dev',
  '##oured',
  'yu',
  '##mmy',
  'fish',
  '.'],
 ['B-ARG0',
  'I-ARG0',
  'I-ARG0',
  'I-ARG0',
  'B-V',
  'I-V',
  'B-ARG1',
  'I-ARG1',
  'I-ARG1',
  'O'])

```

To approach this problem, iterate through each word/label pair in the sentence. Call the tokenizer on the word. This may result in one or more tokens. Create the correct number of labels to match the number of tokens. Take care to not generate multiple B- tokens.


This approach is a bit slower than tokenizing the entire sentence, but is necessary to produce proper input tokenization for the pre-trained BERT model, and the matching target labels.

In [ ]:
def tokenize_with_labels(sentence, text_labels, tokenizer):
    """
    Tokenizes a sentence using the tokenizer while aligning BIO labels with subword tokens.
    Each word may be split into multiple subword tokens, and we must extend the label accordingly:
    - The first subword gets the original label.
    - Any subsequent subwords get the corresponding I- label (or just repeat 'O' if that's the label).
    """
    tokenized_sentence = []
    labels = []

    for word, label in zip(sentence, text_labels):
        subwords = tokenizer.tokenize(word)
        tokenized_sentence.extend(subwords)

        if label == 'O':
            labels.extend(['O'] * len(subwords))
        else:
            prefix, tag_type = label.split('-', 1)
            labels.append(label)
            labels.extend([f'I-{tag_type}'] * (len(subwords) - 1))

    return tokenized_sentence, labels


In [ ]:
tokenize_with_labels("the fancyful penguin devoured yummy fish .".split(), "B-ARG0 I-ARG0 I-ARG0 B-V B-ARG1 I-ARG1 O".split(), tokenizer)

(['the',
  'fancy',
  '##ful',
  'penguin',
  'dev',
  '##oured',
  'yu',
  '##mmy',
  'fish',
  '.'],
 ['B-ARG0',
  'I-ARG0',
  'I-ARG0',
  'I-ARG0',
  'B-V',
  'I-V',
  'B-ARG1',
  'I-ARG1',
  'I-ARG1',
  'O'])

### 1.2 Loading the Dataset

Next, we are creating a PyTorch [Dataset](https://pytorch.org/docs/stable/data.html#torch.utils.data.Dataset) class. This class acts as a contained for the training, development, and testing data in memory. You should already be familiar with Datasets and Dataloaders from homework 3.

1.2.1 **TODO**: Write the \_\_init\_\_(self, filename) method that reads in the data from a data file (specified by the filename).

For each annotation you start with  the tokens in the sentence, and the BIO tags. Then you need to create the following

1. call the `tokenize_with_labels` function to tokenize the sentence.
2. Add the (token, label) pair to the self.items list.

1.2.2 **TODO**: Write the \_\_len\_\_(self) method that returns the total number of items.

1.2.3 **TODO**: Write the \_\_getitem\_\_(self, k) method that returns a single item in a format BERT will understand.
* We need to process the sentence by adding "\[CLS\]" as the first token and "\[SEP\]" as the last token. The need to pad the token sequence to 128 tokens using the "\[PAD\]" symbol. This needs to happen both for the inputs (sentence token sequence) and outputs (BIO tag sequence).
* We need to create an *attention mask*, which is a sequence of 128 tokens indicating the actual input symbols (as a 1) and \[PAD\] symbols (as a 0).
* We need to create a *predicate indicator* mask, which is a sequence of 128 tokens with at most one 1, in the position of the "B-V" tag. All other entries should be 0. The model will use this information to understand where the predicate is located.

* Finally, we need to convert the token and tag sequence into numeric indices. For the tokens, this can be done using the `tokenizer.convert_tokens_to_ids` method. For the tags, use the `role_to_id` dictionary.
Each sequence must be a pytorch tensor of shape (1,128). You can convert a list of integer values like this `torch.tensor(token_ids, dtype=torch.long)`.

To keep everything organized, we will return a dictionary in the following format

```
{'ids': token_tensor,
 'targets': tag_tensor,
 'mask': attention_mask_tensor,
 'pred': predicate_indicator_tensor}
```


(Hint: To debug these, read in the first annotation only / the first few annotations)


In [ ]:
from torch.utils.data import Dataset, DataLoader

class SrlData(Dataset):

    def __init__(self, filename):
        super(SrlData, self).__init__()

        self.max_len = 128
        self.tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased', do_lower_case=True)
        self.items = []

        with open(filename, 'r') as f:
            lines = f.readlines()

        for i in range(0, len(lines), 4):
            if i + 3 >= len(lines):
                continue

            tokens = lines[i+1].strip().split('\t')
            labels = lines[i+3].strip().split('\t')

            if len(tokens) != len(labels):
                continue

            tokens, labels = tokenize_with_labels(tokens, labels, self.tokenizer)
            self.items.append((tokens, labels))
    def __len__(self):
        return len(self.items)

    def __getitem__(self, k):
        tokens, tags = self.items[k]

        tokens = ['[CLS]'] + tokens + ['[SEP]']
        tags = ['O'] + tags + ['O']

        padding_length = self.max_len - len(tokens)
        if padding_length > 0:
            tokens += ['[PAD]'] * padding_length
            tags += ['O'] * padding_length
        else:
            tokens = tokens[:self.max_len]
            tags = tags[:self.max_len]

        token_ids = self.tokenizer.convert_tokens_to_ids(tokens)
        tag_ids = [role_to_id.get(tag, 0) for tag in tags]

        attn_mask = [1 if tok != '[PAD]' else 0 for tok in tokens]

        pred_mask = [1 if tag == 'B-V' else 0 for tag in tags]

        return {
            'ids': torch.tensor(token_ids, dtype=torch.long),
            'mask': torch.tensor(attn_mask, dtype=torch.long),
            'targets': torch.tensor(tag_ids, dtype=torch.long),
            'pred': torch.tensor(pred_mask, dtype=torch.long)
        }

In [ ]:
data = SrlData("propbank_train.tsv")

## 2. Model Definition

In [ ]:
from torch.nn import Module, Linear, CrossEntropyLoss
from transformers import BertModel
import torch.nn as nn

We will define the pyTorch model as a subclass of the [torch.nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html) class. The code for the model is provided for you. It may help to take a look at the documentation to remind you of how Module works. Take a look at how the huggingface BERT model simply becomes another sub-module.

In [ ]:
class SrlModel(nn.Module):
    def __init__(self, role_to_id):
        super(SrlModel, self).__init__()

        self.encoder = BertModel.from_pretrained("bert-base-uncased")


        self.classifier = nn.Linear(768, len(role_to_id))

    def forward(self, input_ids, attn_mask, pred_indicator):

        bert_output = self.encoder(
            input_ids=input_ids,
            attention_mask=attn_mask,
            token_type_ids=pred_indicator
        )

        enc_tokens = bert_output.last_hidden_state

        logits = self.classifier(enc_tokens)

        return logits

In [ ]:
model = SrlModel(role_to_id).to('cuda') # create new model and store weights in GPU memory

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Now we are ready to try running the model with just a single input example to check if it is working correctly. Clearly it has not been trained, so the output is not what we expect. But we can see what the loss looks like for an initial sanity check.

**TODO**:
* Take a single data item from the dev set, as provided by your Dataset class defined above. Obtain the input token ids, attention mask, predicate indicator mask, and target labels.
* Run the model on the ids, attention mask, and predicate mask like this:

In [ ]:
dev_data = SrlData("propbank_dev.tsv")
sample = dev_data[0]

ids = sample['ids'].unsqueeze(0).to('cuda')           # shape: (1, 128)
mask = sample['mask'].unsqueeze(0).to('cuda')         # shape: (1, 128)
pred = sample['pred'].unsqueeze(0).to('cuda')         # shape: (1, 128)
targets = sample['targets'].unsqueeze(0).to('cuda')   # shape: (1, 128)

outputs = model(ids, mask, pred)                      # shape: (1, 128, num_labels)

criterion = torch.nn.CrossEntropyLoss(ignore_index=-100)
loss = criterion(outputs.view(-1, outputs.shape[-1]), targets.view(-1))

print("Initial loss:", loss.item())


Initial loss: 4.1558051109313965


**TODO**:
Compute the loss on this one item only.
The initial loss should be close to -ln(1/num_labels)

Without training we would assume that all labels for each token (including the target label) are equally likely, so the negative log probability for the targets should be approximately $$-\ln(\frac{1}{\text{num_labels}}).$$ This is what the loss function should return on a single example. This is a good sanity check to run for any multi-class prediction problem.

In [ ]:
import math
-math.log(1 / len(role_to_id), math.e)

3.970291913552122

In [ ]:
loss_function = CrossEntropyLoss(ignore_index = -100, reduction='mean')

# complete this. Note that you still have to provide a (batch_size, input_pos)
# tensor for each parameter, where batch_size =1

# outputs = model(ids, mask, pred)
# loss = loss_function(...)
# loss.item()   #this should be approximately the score from the previous cell

# Compute loss
loss = loss_function(outputs.view(-1, outputs.shape[-1]), targets.view(-1))

print("Initial loss:", loss.item())

num_labels = outputs.shape[-1]
print("Expected untrained loss ≈ ln(num_labels) =", math.log(num_labels))


Initial loss: 4.1558051109313965
Expected untrained loss ≈ ln(num_labels) = 3.970291913552122


**TODO**: At this point you should also obtain the actual predictions by taking the argmax over each position.
The result should look something like this (values will differ).

```
tensor([[ 1,  4,  4,  4,  4,  4,  5, 29, 29, 29,  4, 28,  6, 32, 32, 32, 32, 32,
         32, 32, 30, 30, 32, 30, 32,  4, 32, 32, 30,  4, 49,  4, 49, 32, 30,  4,
         32,  4, 32, 32,  4,  2,  4,  4, 32,  4, 32, 32, 32, 32, 30, 32, 32, 30,
         32,  4,  4, 49,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  6,  6, 32, 32,
         30, 32, 32, 32, 32, 32, 30, 30, 30, 32, 30, 49, 49, 32, 32, 30,  4,  4,
          4,  4, 29,  4,  4,  4,  4,  4,  4, 32,  4,  4,  4, 32,  4, 30,  4, 32,
         30,  4, 32,  4,  4,  4,  4,  4, 32,  4,  4,  4,  4,  4,  4,  4,  4,  4,
          4,  4]], device='cuda:0')
```

Then use the id_to_role dictionary to decode to actual tokens.

```
['[CLS]', 'O', 'O', 'O', 'O', 'O', 'B-ARG0', 'I-ARG0', 'I-ARG0', 'I-ARG0', 'O', 'B-V', 'B-ARG1', 'I-ARG2', 'I-ARG2', 'I-ARG2', 'I-ARG2', 'I-ARG2', 'I-ARG2', 'I-ARG2', 'I-ARG1', 'I-ARG1', 'I-ARG2', 'I-ARG1', 'I-ARG2', 'O', 'I-ARG2', 'I-ARG2', 'I-ARG1', 'O', 'I-ARGM-TMP', 'O', 'I-ARGM-TMP', 'I-ARG2', 'I-ARG1', 'O', 'I-ARG2', 'O', 'I-ARG2', 'I-ARG2', 'O', '[SEP]', 'O', 'O', 'I-ARG2', 'O', 'I-ARG2', 'I-ARG2', 'I-ARG2', 'I-ARG2', 'I-ARG1', 'I-ARG2', 'I-ARG2', 'I-ARG1', 'I-ARG2', 'O', 'O', 'I-ARGM-TMP', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ARG1', 'B-ARG1', 'I-ARG2', 'I-ARG2', 'I-ARG1', 'I-ARG2', 'I-ARG2', 'I-ARG2', 'I-ARG2', 'I-ARG2', 'I-ARG1', 'I-ARG1', 'I-ARG1', 'I-ARG2', 'I-ARG1', 'I-ARGM-TMP', 'I-ARGM-TMP', 'I-ARG2', 'I-ARG2', 'I-ARG1', 'O', 'O', 'O', 'O', 'I-ARG0', 'O', 'O', 'O', 'O', 'O', 'O', 'I-ARG2', 'O', 'O', 'O', 'I-ARG2', 'O', 'I-ARG1', 'O', 'I-ARG2', 'I-ARG1', 'O', 'I-ARG2', 'O', 'O', 'O', 'O', 'O', 'I-ARG2', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
```

For now, just make sure you understand how to do this for a single example. Later, you will write a more formal function to do this once we have trained the model.

In [ ]:
outputs = model(ids, mask, pred)  # shape: (1, 128, num_labels)

predicted_ids = torch.argmax(outputs, dim=-1)  # shape: (1, 128)
print("Predicted IDs:", predicted_ids)

predicted_ids_list = predicted_ids.squeeze(0).tolist()
decoded_labels = [id_to_role.get(idx, '[PAD]') for idx in predicted_ids_list]
print("Decoded labels:", decoded_labels)

tokens = dev_data.tokenizer.convert_ids_to_tokens(sample['ids'])
print("Tokens:", tokens)


Predicted IDs: tensor([[ 7,  9,  9,  7,  7, 15, 15, 34, 34, 44,  7, 27, 15,  9, 15, 22,  9, 32,
         34, 32, 34, 45, 34, 34, 22, 22, 27,  9,  7, 47, 44, 39, 39, 39, 39, 34,
         39, 39, 34, 34, 15, 15, 15, 15, 39, 34, 39, 39, 34, 15, 34, 34, 34, 34,
         39, 34,  7,  9, 15, 15, 27, 27, 39, 34, 34, 34, 27, 15, 15, 27, 15, 44,
         39, 34, 27, 34, 34, 34, 34, 34, 34, 20,  9, 34, 39, 39,  9,  9,  9, 27,
         34, 39, 34, 34, 34, 27, 15, 15, 15, 27, 39, 39, 39, 34, 15, 34, 34, 34,
         34, 34,  9, 20,  9,  9, 44, 39, 39, 44, 20, 39, 39, 39, 39, 39, 39, 39,
         39, 39]], device='cuda:0')
Decoded labels: ['B-ARG1-DSP', 'B-ARG3', 'B-ARG3', 'B-ARG1-DSP', 'B-ARG1-DSP', 'B-ARGM-DIS', 'B-ARGM-DIS', 'I-ARG4', 'I-ARG4', 'I-ARGM-MOD', 'B-ARG1-DSP', 'B-ARGM-LVB', 'B-ARGM-DIS', 'B-ARG3', 'B-ARGM-DIS', 'B-ARGM-PRD', 'B-ARG3', 'I-ARG2', 'I-ARG4', 'I-ARG2', 'I-ARG4', 'I-ARGM-NEG', 'I-ARG4', 'I-ARG4', 'B-ARGM-PRD', 'B-ARGM-PRD', 'B-ARGM-LVB', 'B-ARG3', 'B-ARG1-DSP', 'I-ARGM-PRP

## 3. Training loop

pytorch provides a DataLoader class that can be wrapped around a Dataset to easily use the dataset for training. The DataLoader allows us to easily adjust the batch size and shuffle the data.

In [ ]:
from torch.utils.data import DataLoader
loader = DataLoader(data, batch_size = 32, shuffle = True)

The following cell contains the main training loop. The code should work as written and report the loss after each batch,
cumulative average loss after each 100 batches, and print out the final average loss after the epoch.

**TODO**: Modify the training loop belowso that it also computes the accuracy for each batch and reports the
average accuracy after the epoch.
The accuracy is the number of correctly predicted token labels out of the number of total predictions.
Make sure you exclude [PAD] tokens, i.e. tokens for which the target label is -100. It's okay to include [CLS] and [SEP] in the accuracy calculation.

In [ ]:
from torch.optim import AdamW

LEARNING_RATE = 1e-5
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

from torch.nn import CrossEntropyLoss

loss_function = CrossEntropyLoss(ignore_index=-100, reduction='mean')

device = 'cuda'

def train():
    """
    Train the model for one epoch and report average loss and accuracy.
    """
    tr_loss = 0
    nb_tr_examples, nb_tr_steps = 0, 0
    total_correct = 0
    total_count = 0

    model.train()

    for idx, batch in enumerate(loader):
        ids = batch['ids'].to(device, dtype=torch.long)
        mask = batch['mask'].to(device, dtype=torch.long)
        targets = batch['targets'].to(device, dtype=torch.long)
        pred_mask = batch['pred'].to(device, dtype=torch.long)

        logits = model(input_ids=ids, attn_mask=mask, pred_indicator=pred_mask)

        loss = loss_function(logits.transpose(2, 1), targets)
        tr_loss += loss.item()

        print("Batch loss:", loss.item())

        nb_tr_steps += 1
        nb_tr_examples += targets.size(0)

        if idx % 100 == 0:
            curr_avg_loss = tr_loss / nb_tr_steps

            print(f"Current average loss: {curr_avg_loss}")

        predictions = torch.argmax(logits, dim=2)
        mask_valid = targets != -100

        correct = (predictions == targets) & mask_valid
        total_correct += correct.sum().item()
        total_count += mask_valid.sum().item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    epoch_loss = tr_loss / nb_tr_steps
    epoch_acc = total_correct / total_count if total_count > 0 else 0.0
    print(f"Training loss epoch: {epoch_loss:.4f}")
    print(f"Training accuracy epoch: {epoch_acc:.4f}")


Now let's train the model for one epoch. This will take a while (up to a few hours).

In [ ]:
train()

Batch loss: 0.11113239079713821
Current average loss: 0.11113239079713821
Batch loss: 0.08432710915803909
Batch loss: 0.08501538634300232
Batch loss: 0.10583503544330597
Batch loss: 0.07005871832370758
Batch loss: 0.08751504123210907
Batch loss: 0.169678196310997
Batch loss: 0.09308133274316788
Batch loss: 0.1112927794456482
Batch loss: 0.0851673111319542
Batch loss: 0.057891447097063065
Batch loss: 0.08513797074556351
Batch loss: 0.09478621929883957
Batch loss: 0.08067075908184052
Batch loss: 0.10312038660049438
Batch loss: 0.10016444325447083
Batch loss: 0.0902305394411087
Batch loss: 0.06359171122312546
Batch loss: 0.06699559837579727
Batch loss: 0.06936521828174591
Batch loss: 0.06288128346204758
Batch loss: 0.07762999087572098
Batch loss: 0.08566196262836456
Batch loss: 0.10619287192821503
Batch loss: 0.07820742577314377
Batch loss: 0.06775528937578201
Batch loss: 0.07756955921649933
Batch loss: 0.07551786303520203
Batch loss: 0.07604039460420609
Batch loss: 0.03590195253491402
Ba

KeyboardInterrupt: 

In my experiments, I found that two epochs are needed for good performance.

In [ ]:
train()

I ended up with a training loss of about 0.19 and a training accuracy of 0.94. Specific values may differ.

At this point, it's a good idea to save the model (or rather the parameter dictionary) so you can continue evaluating the model without having to retrain.

In [ ]:
torch.save(model.state_dict(), "srl_model_fulltrain_2epoch_finetune_1e-05.pt")

## 4. Decoding

In [ ]:
# Optional step: If you stopped working after part 3, first load the trained model

model = SrlModel().to('cuda')
model.load_state_dict(torch.load("srl_model_fulltrain_2epoch_finetune_1e-05.pt"))
model = model.to('cuda')

**TODO (this is the fun part)**: Now that we have a trained model, let's try labeling an unseen example sentence. Complete the functions decode_output and label_sentence below. decode_output takes the logits returned by the model, extracts the argmax to obtain the label predictions for each token, and then translate the result into a list of string labels.

label_sentence takes a list of input tokens and a predicate index, prepares the model input, call the model and then call decode_output to produce a final result.

Note that you have already implemented all components necessary (preparing the input data from the token list and predicate index, decoding the model output). But now you are putting it together in one convenient function.

In [ ]:
tokens = "A U. N. team spent an hour inside the hospital , where it found evident signs of shelling and gunfire .".split()

In [ ]:
def decode_output(logits):
    """
    Given the model output logits, return predicted string labels.
    """
    predicted_ids = torch.argmax(logits, dim=-1).squeeze(0).tolist()  # shape: [seq_len]
    labels = [id_to_role.get(idx, '[PAD]') for idx in predicted_ids]
    return labels

def label_sentence(tokens, pred_idx):
    """
    Labels the input tokens using the model and aligns predictions to original words.
    Returns a list of (original_token, predicted_label).
    """
    max_len = 128

    encoding = tokenizer(tokens,
                         is_split_into_words=True,
                         return_tensors='pt',
                         padding='max_length',
                         truncation=True,
                         max_length=max_len,
                         return_attention_mask=True)

    input_ids = encoding['input_ids'].to('cuda')
    attn_mask = encoding['attention_mask'].to('cuda')
    word_ids = encoding.word_ids(batch_index=0)  # maps subwords to original tokens

    pred_mask = [(1 if word_idx == pred_idx else 0) if word_idx is not None else 0 for word_idx in word_ids]
    pred_mask = torch.tensor(pred_mask, dtype=torch.long).unsqueeze(0).to('cuda')

    with torch.no_grad():
        logits = model(input_ids, attn_mask, pred_mask)

    subword_labels = decode_output(logits)

    token_to_label = {}
    for i, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue
        if word_idx not in token_to_label:
            token_to_label[word_idx] = subword_labels[i]

    aligned_output = [(tokens[i], token_to_label.get(i, 'O')) for i in range(len(tokens))]
    return aligned_output


In [ ]:
predicate_index = tokens.index("found")

results = label_sentence(tokens, predicate_index)

for token, label in results:
    print(f"('{token}', '{label}'),")


('A', 'O'),
('U.', 'O'),
('N.', 'O'),
('team', 'O'),
('spent', 'O'),
('an', 'O'),
('hour', 'O'),
('inside', 'O'),
('the', 'O'),
('hospital', 'I-ARGM-LOC'),
(',', 'O'),
('where', 'B-ARGM-LOC'),
('it', 'B-ARG0'),
('found', 'B-V'),
('evident', 'B-ARG1'),
('signs', 'I-ARG1'),
('of', 'I-ARG1'),
('shelling', 'I-ARG1'),
('and', 'I-ARG1'),
('gunfire', 'I-ARG1'),
('.', 'O'),


The expected output is somethign like this:
```   
 ('A', 'O'),
 ('U.', 'O'),
 ('N.', 'O'),
 ('team', 'O'),
 ('spent', 'O'),
 ('an', 'O'),
 ('hour', 'O'),
 ('inside', 'O'),
 ('the', 'B-ARGM-LOC'),
 ('hospital', 'I-ARGM-LOC'),
 (',', 'O'),
 ('where', 'B-ARGM-LOC'),
 ('it', 'B-ARG0'),
 ('found', 'B-V'),
 ('evident', 'B-ARG1'),
 ('signs', 'I-ARG1'),
 ('of', 'I-ARG1'),
 ('shelling', 'I-ARG1'),
 ('and', 'I-ARG1'),
 ('gunfire', 'I-ARG1'),
 ('.', 'O'),
```


### 5. Evaluation 1: Token-Based Accuracy
We want to evaluate the model on the dev or test set.

In [ ]:
dev_data = SrlData("propbank_dev.tsv") # Takes a while because we preprocess all data offline

In [ ]:
from torch.utils.data import DataLoader
loader = DataLoader(dev_data, batch_size = 1, shuffle = False)

In [ ]:
# Optional: Load the model again if you stopped working prior to this step.
# model = SrlModel()
# model.load_state_dict(torch.load("srl_model_fulltrain_2epoch_finetune_1e-05.pt"))
# model = mode.to('cuda')

**TODO**: Complete the evaluate_token_accuracy function below. The function should iterate through the items in the data loader (see training loop in part 3). Run the model on each sentence/predicate pair and extract the predictions.

For each sentence, count the correct predictions and the total predictions. Finally, compute the accuracy as #correct_predictions / #total_predictions

Careful: You need to filter out the padded positions ([PAD] target tokens), as well as [CLS] and [SEP]. It's okay to include [B-V] in the count though.

In [ ]:
def evaluate_token_accuracy(model, loader):
    model.eval()  # set model to evaluation mode

    total_correct = 0
    total_predictions = 0

    with torch.no_grad():
        for batch in loader:
            ids = batch['ids'].to('cuda')
            mask = batch['mask'].to('cuda')
            targets = batch['targets'].to('cuda')
            pred = batch['pred'].to('cuda')

            logits = model(ids, mask, pred)
            predictions = torch.argmax(logits, dim=-1)

            predictions = predictions.view(-1)
            targets = targets.view(-1)
            masks = mask.view(-1)

            valid = (targets != -100) & (predictions != -100)

            total_correct += (predictions[valid] == targets[valid]).sum().item()
            total_predictions += valid.sum().item()

    acc = total_correct / total_predictions if total_predictions > 0 else 0.0
    print(f"Token-level Accuracy: {acc:.4f}")


In [ ]:
from torch.utils.data import DataLoader

dev_data = SrlData("propbank_dev.tsv")
dev_loader = DataLoader(dev_data, batch_size=16, shuffle=False)

evaluate_token_accuracy(model, dev_loader)


Token-level Accuracy: 0.9774


### 6. Span-Based evaluation

While the accuracy score in part 5 is encouraging, an accuracy-based evaluation is problematic for two reasons. First, most of the target labels are actually O. Second, it only tells us that per-token prediction works, but does not directly evaluate the SRL performance.

Instead, SRL systems are typically evaluated on micro-averaged precision, recall, and F1-score for predicting labeled spans.

More specifically, for each sentence/predicate input, we run the model, decode the output, and extract a set of labeled spans (from the output and the target labels). These spans are (i,j,label) tuples.  

We then compute the true_positives, false_positives, and false_negatives based on these spans.

In the end, we can compute

* Precision:  true_positive / (true_positives + false_positives)  , that is the number of correct spans out of all predicted spans.

* Recall: true_positives / (true_positives + false_negatives) , that is the number of correct spans out of all target spans.

* F1-score:   (2 * precision * recall) / (precision + recall)


For example, consider

| |[CLS]|The|judge|scheduled|to|preside|over|his|trial|was|removed|from|the|case|today|.|             
|--||---|-----|---------|--|-------|----|---|-----|---|-------|----|---|----|-----|-|             
||0|1|2|3|4|5|6|7|8|9|1O|11|12|13|14|15|
|target|[CLS]|B-ARG1|I-ARG1|B-V|B-ARG2|I-ARG2|I-ARG2|I-ARG2|I-ARG2|O|O|O|O|O|O|O|
|prediction|[CLS]|B-ARG1|I-ARG1|B-V|I-ARG2|I-ARG2|O|O|O|O|O|O|O|O|B-ARGM-TMP|O|

The target spans are (1,2,"ARG1"), and (4,8,"ARG2").

The predicted spans would be (1,2,"ARG1"), (14,14,"ARGM-TMP"). Note that in the prediction, there is no proper ARG2 span because we are missing the B-ARG2 token, so this span should not be created.

So for this sentence we woudl get: true_positives: 1 false_positives: 1 false_negatives: 1

*TODO*: Complete the function evaluate_spans that performs the span-based evaluation on the given model and data loader. You can use the provided extract_spans function, which returns the spans as a dictionary. For example
{(1,2): "ARG1", (4,8):"ARG2"}

In [ ]:
def extract_spans(labels):
    spans = {} # map (start,end) ids to label
    current_span_start = 0
    current_span_type = ""
    inside = False
    for i, label in enumerate(labels):
        if label.startswith("B"):
            if inside:
                if current_span_type != "V":
                    spans[(current_span_start,i)] = current_span_type
            current_span_start = i
            current_span_type = label[2:]
            inside = True
        elif inside and label.startswith("O"):
            if current_span_type != "V":
                spans[(current_span_start,i)] = current_span_type
            inside = False
        elif inside and label.startswith("I") and label[2:] != current_span_type:
            if current_span_type != "V":
                spans[(current_span_start,i)] = current_span_type
            inside = False
    return spans


In [ ]:
def evaluate_spans(model, loader):
    model.eval()

    total_tp = 0
    total_fp = 0
    total_fn = 0

    with torch.no_grad():
        for idx, batch in enumerate(loader):
            ids = batch['ids'].to('cuda')
            mask = batch['mask'].to('cuda')
            targets = batch['targets'].to('cuda')
            pred_mask = batch['pred'].to('cuda')

            logits = model(ids, mask, pred_mask)
            predictions = torch.argmax(logits, dim=-1)

            for i in range(ids.shape[0]):
                pred_ids = predictions[i].tolist()
                true_ids = targets[i].tolist()

                # Convert to string labels
                pred_labels = [id_to_role.get(idx, 'O') for idx in pred_ids]
                true_labels = [id_to_role.get(idx, 'O') for idx in true_ids]

                # Mask out [PAD], [CLS], and [SEP] — treat them as 'O'
                for j, label in enumerate(pred_labels):
                    if label in ['[PAD]', '[CLS]', '[SEP]']:
                        pred_labels[j] = 'O'
                for j, label in enumerate(true_labels):
                    if label in ['[PAD]', '[CLS]', '[SEP]']:
                        true_labels[j] = 'O'

                # Extract spans
                pred_spans = extract_spans(pred_labels)
                gold_spans = extract_spans(true_labels)

                pred_items = set((start, end, label) for (start, end), label in pred_spans.items())
                gold_items = set((start, end, label) for (start, end), label in gold_spans.items())

                total_tp += len(pred_items & gold_items)
                total_fp += len(pred_items - gold_items)
                total_fn += len(gold_items - pred_items)

    total_p = total_tp / (total_tp + total_fp + 1e-8)
    total_r = total_tp / (total_tp + total_fn + 1e-8)
    total_f = (2 * total_p * total_r) / (total_p + total_r + 1e-8)

    print(f"Overall P: {total_p:.4f}  Overall R: {total_r:.4f}  Overall F1: {total_f:.4f}")


In [ ]:
evaluate_spans(model, dev_loader)

Overall P: 0.6928  Overall R: 0.7407  Overall F1: 0.7160


In my evaluation, I got an F score of 0.82  (which slightly below the state-of-the art in 2018)

### OPTIONAL:

Repeat the span-based evaluation, but print out precision/recall/f1-score for each role separately.